# 02 – Preprocessing & Data Cleaning

### Purpose of the Notebook
This notebook applies systematic cleaning and standardisation to the pre‑saved datasets (dataset.pkl and dataset_de.pkl).
All decisions are based on the insights from Notebook 01_data_overview (EDA), including handling of missing data, removal of low‑quality fields, type corrections, logical consistency checks, and creation of derived features.

### Steps
- Load pre‑saved datasets
- Apply preprocessing pypline
- Save cleaned datasets

--------------
### Load Pre-Saved Dataset

-----------

In [1]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

In [2]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [4]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.preprocessing import preprocess
from my_scripts.eda import overview

In [5]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset.pkl")
df_de = pd.read_pickle("../data/dataset_de.pkl")

print("EU dataset:", df.shape)
print("DE dataset:", df_de.shape)

EU dataset: (4362869, 75)
DE dataset: (309652, 75)


--------------
### Apply preprocessing pipeline
-----------

In [6]:
# ---------------------------------------------------------
# full EU dataset
# ---------------------------------------------------------

df_clean = preprocess(df)


In [7]:
# shape of the cleaned EU dataset
print(df_clean.shape)

(4362869, 30)


In [8]:
# inspekt of the cleaned EU dataset
overview(df_clean)

,dtype,total,missing_n,missing_%,uniques_n,uniques
YEAR,int64,4362869,0,0.00,9,"[2008, 2009, 2010, 2011, 2012, 2013, 2014, 201..."
ID_TYPE,int64,4362869,0,0.00,7,"[3, 6, 18, 25, 21, 23, 22]"
DT_DISPATCH,datetime64[ns],4362869,0,0.00,3297,"[2007-12-11 00:00:00, 2007-12-27 00:00:00, 200..."
XSD_VERSION,object,4362869,0,0.00,7,"[D205, D206, D207, R207.S3, R208.S1, R208.S2, ..."
CANCELLED,int64,4362869,0,0.00,2,"[0, 1]"
CORRECTIONS,int64,4362869,0,0.00,10,"[0, 1, 2, 4, 3, 5, 84, 7, 8, 6]"
ISO_COUNTRY_CODE,object,4362869,0,0.00,33,"[DE, FR, ES, SE, PL, IT, HU, CY, UK, RO, PT, N..."
CAE_TYPE,object,4362869,0,0.00,10,"[8, 3, 6, 1, R, N, 4, 5A, 5, Z]"
B_AWARDED_BY_CENTRAL_BODY,object,4362869,0,0.00,3,"[nan, Y, N]"
TYPE_OF_CONTRACT,object,4362869,0,0.00,3,"[W, U, S]"


In [9]:
# ---------------------------------------------------------
# dataset for Germany
# ---------------------------------------------------------

df_de_clean = preprocess(df_de)

In [10]:
# shape of the cleaned dataset for Germany
print(df_de_clean.shape)

(309652, 30)


In [11]:
# inspekt of the cleaned dataset  for Germany
overview(df_de_clean)

,dtype,total,missing_n,missing_%,uniques_n,uniques
YEAR,int64,309652,0,0.00,9,"[2008, 2009, 2010, 2011, 2012, 2013, 2014, 201..."
ID_TYPE,int64,309652,0,0.00,5,"[3, 6, 18, 25, 21]"
DT_DISPATCH,datetime64[ns],309652,0,0.00,2805,"[2007-12-11 00:00:00, 2007-12-27 00:00:00, 200..."
XSD_VERSION,object,309652,0,0.00,7,"[D205, D206, D207, R207.S3, R208.S1, R208.S2, ..."
CANCELLED,int64,309652,0,0.00,2,"[0, 1]"
CORRECTIONS,int64,309652,0,0.00,3,"[0, 1, 2]"
ISO_COUNTRY_CODE,object,309652,0,0.00,1,[DE]
CAE_TYPE,object,309652,0,0.00,10,"[8, 3, 6, N, 1, R, 4, 5A, 5, Z]"
B_AWARDED_BY_CENTRAL_BODY,object,309652,0,0.00,3,"[nan, Y, N]"
TYPE_OF_CONTRACT,object,309652,0,0.00,3,"[W, U, S]"


#### Notes: Summary of Data Cleaning Results

1. Dataset Size After Preprocessing
- Full EU dataset:
  - Before: 4,362,869 rows × 75 columns
  - After: 4,362,869 rows × 30 columns

- Germany-only dataset:
  - Before: 309,652 rows × 75 columns
  - After: 309,652 rows × 30 columns

2. The reduction from 75 to 30 columns is the result of removing:
- Columns removed due to >40% missing values
  - Winner information (WIN_*)
  - Contracting authority details (CAE_*)
  - GPA-related fields
  - Secondary financial fields (VALUE_EURO_FIN_*, AWARD_VALUE_EURO_FIN_1)
  - Award criteria weights (CRIT_*)
  - Additional CPVs
  - High-cardinality procedural flags (B_MULTIPLE_, B_FRA_, FRA_ESTIMATED, etc.)
  - TED_NOTICE_URL

- Columns removed due to irrelevance for competition modelling
  - Identifiers (ID_NOTICE_CAN, ID_AWARD, ID_LOT_AWARDED, CONTRACT_NUMBER)
  - Textual descriptions (TITLE)
  - Non-award information (INFO_ON_NON_AWARD, INFO_UNPUBLISHED)
  - Administrative metadata (MAIN_ACTIVITY, EU_INST_CODE)

3. Key Variables Retained Despite Missing Values
Several columns with substantial missingness were intentionally retained because they are critical for modelling tender competition and failure risk:
- VALUE_EURO
  - Missing: 39.85% (EU dataset)
  - Missing: 50.28% (Germany)
  - Importance: baseline contract value, used for log-transformations and value bins.

- AWARD_VALUE_EURO
  - Missing: 34.41% (EU dataset)
  - Missing: 51.26% (Germany)
  - Importance: actual awarded value, essential for understanding tender dynamics.

- NUMBER_OFFERS
  - Missing: 18.94% (EU dataset)
  - Missing: 16.73% (Germany)
  - Importance: primary target variable for defining failed / low-competition tenders.

These variables were not removed because:
They are central to the analytical goal (competition modelling).
Missingness is informative, not random.

4. Additional Preprocessing Steps
- All categorical missing values were replaced with "Unknown".
- Numeric missing values were left as NaN, to be handled during modelling.

5. Conversion of Categorical Variables
To ensure correct feature engineering and avoid dtype-related errors, all categorical variables were explicitly converted to string (object).
This includes fields such as:
  - ISO_COUNTRY_CODE
  - CAE_TYPE
  - TYPE_OF_CONTRACT
  - TAL_LOCATION_NUTS
  - B_EU_FUNDS
  - TOP_TYPE
  - CRIT_CODE
  - B_ELECTRONIC_AUCTION
  - B_AWARDED_TO_A_GROUP
  - WIN_COUNTRY_CODE
  - B_CONTRACTOR_SME
  - B_SUBCONTRACTED
  - CPV (converted from float → string, with removal of “.0”)
This guarantees correct handling of categorical data and enables creation of CPV hierarchy features in the feature engineering stage.

--------------
### Save cleaned datasets

-----------

In [12]:
df_clean.to_pickle("../data/dataset_clean.pkl")
df_de_clean.to_pickle("../data/dataset_de_clean.pkl")